# Phase 1: Train Shared NAR on CLRS-30 Graph Algorithms

Train a single MPNN-based NAR model on multiple graph algorithms simultaneously.
The processor is shared across all algorithms; the encoder/decoder adapt dynamically
based on each algorithm's output/hint types.

**Algorithms:** BFS, DFS, Dijkstra, Bellman-Ford, MST Prim,
Articulation Points, Bridges, Fast MIS, Eccentricity

In [9]:
!nvidia-smi

Wed Apr  1 05:38:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             47W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [10]:
# --- Environment setup (Colab + local) ---
import os, subprocess, sys

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(os.environ.get("COLAB_RELEASE_TAG", ""))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_DIR = "/content/nar-mechinterp"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/aniervs/nar-mechinterp.git", REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    os.chdir(REPO_DIR)

    # Install only the deps Colab doesn't ship (torch, numpy, matplotlib, etc. are preinstalled)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "salsa-clrs @ git+https://github.com/jkminder/SALSA-CLRS.git",
                    "torch-geometric", "dm-haiku", "einops", "hydra-core", "omegaconf",
                    "rich", "scikit-learn", "plotly", "seaborn"], check=True)

    # Add repo root to Python path so our packages are importable
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Patch salsa-clrs to enable extra algorithms
    subprocess.run([sys.executable, "scripts/patch_salsaclrs.py"], check=True)
    print(f"Working directory: {os.getcwd()}")
    print("Colab setup complete.")
else:
    if os.path.basename(os.getcwd()) == "experiments":
        os.chdir("..")
    print(f"Running locally from {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/nar-mechinterp
Colab setup complete.


In [11]:
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from data import (
    AVAILABLE_ALGORITHMS,
    get_clrs_dataloader,
    get_algorithm_spec,
    spec_to_model_types,
    batch_to_model_inputs,
    MultiAlgorithmLoader,
)
from models import NARModel
from utils.clrs_metrics import evaluate_outputs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)

Device: cuda


## Configuration

In [12]:
# --- Algorithm selection ---
ALGORITHMS = AVAILABLE_ALGORITHMS
print(f"Training on {len(ALGORITHMS)} algorithms: {ALGORITHMS}")

# === Scale toggle ===
LOCAL_DEBUG = False

if LOCAL_DEBUG:
    HIDDEN_DIM = 32
    NUM_LAYERS = 2
    NUM_HEADS = 4
    NAR_EPOCHS = 10
    TRAIN_SAMPLES = 100
    VAL_SAMPLES = 32
else:
    HIDDEN_DIM = 128
    NUM_LAYERS = 4
    NUM_HEADS = 8
    NAR_EPOCHS = 100
    TRAIN_SAMPLES = 1000
    VAL_SAMPLES = 128

PROCESSOR_TYPE = "mpnn"
NAR_BATCH_SIZE = 32
NAR_LR = 1e-3
NUM_NODES = 16
EDGE_PROB = 0.2
OOD_NUM_NODES = 64  # for OOD evaluation

# Paths — use Google Drive on Colab for persistence
if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/nar-mechinterp")
    CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "multi_algo"
else:
    CHECKPOINT_DIR = Path("checkpoints") / "multi_algo"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mode: {'LOCAL DEBUG' if LOCAL_DEBUG else 'FULL SCALE'}")
print(f"NAR: hidden_dim={HIDDEN_DIM}, layers={NUM_LAYERS}, heads={NUM_HEADS}")
print(f"Training: {NAR_EPOCHS} epochs, {TRAIN_SAMPLES} samples/algo")
print(f"OOD test: {OOD_NUM_NODES} nodes")
print(f"Checkpoints: {CHECKPOINT_DIR}")

Training on 9 algorithms: ['bfs', 'dfs', 'dijkstra', 'bellman_ford', 'mst_prim', 'articulation_points', 'bridges', 'fast_mis', 'eccentricity']
Mode: FULL SCALE
NAR: hidden_dim=128, layers=4, heads=8
Training: 100 epochs, 1000 samples/algo
Checkpoints: /content/drive/MyDrive/nar-mechinterp/checkpoints/multi_algo


## Load Datasets

Create a `MultiAlgorithmLoader` for train and val splits. Each yields
`(batch, algo_name, output_types, hint_types)` in round-robin order.

In [13]:
train_loader = MultiAlgorithmLoader(
    algorithms=ALGORITHMS,
    split="train",
    batch_size=NAR_BATCH_SIZE,
    num_samples=TRAIN_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED,
    shuffle=True,
)

val_loader = MultiAlgorithmLoader(
    algorithms=ALGORITHMS,
    split="val",
    batch_size=NAR_BATCH_SIZE,
    num_samples=VAL_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 1,
    shuffle=False,
)

print(f"Train: {len(train_loader)} total batches across {train_loader.num_algorithms} algorithms")
print(f"Val:   {len(val_loader)} total batches")
print(f"\nPer-algorithm specs:")
for algo in ALGORITHMS:
    ot, ht = train_loader.output_types[algo], train_loader.hint_types[algo]
    print(f"  {algo:25s} outputs={list(ot.keys()):30s} hints={list(ht.keys())}")

Processing...
2026-04-01 05:38:43.010 | INFO     | salsaclrs.data:process:326 - Generating bfs dataset with 1000 samples on -1 cores
2026-04-01 05:38:43.011 | INFO     | salsaclrs.data:process:327 - Graph generator: er with kwargs {'n': 16, 'p': 0.2}


  0%|          | 0/1000 [00:00<?, ?it/s]

Done!
Processing...
2026-04-01 05:38:45.648 | INFO     | salsaclrs.data:process:326 - Generating dfs dataset with 1000 samples on -1 cores
2026-04-01 05:38:45.649 | INFO     | salsaclrs.data:process:327 - Graph generator: er with kwargs {'n': 16, 'p': 0.2}


  0%|          | 0/1000 [00:00<?, ?it/s]

Done!
Processing...
2026-04-01 05:38:56.599 | INFO     | salsaclrs.data:process:326 - Generating dijkstra dataset with 1000 samples on -1 cores
2026-04-01 05:38:56.600 | INFO     | salsaclrs.data:process:327 - Graph generator: er with kwargs {'n': 16, 'p': 0.2}


...

Done!
Processing...
2026-04-01 05:40:08.145 | INFO     | salsaclrs.data:process:326 - Generating eccentricity dataset with 128 samples on -1 cores
2026-04-01 05:40:08.146 | INFO     | salsaclrs.data:process:327 - Graph generator: er with kwargs {'n': 16, 'p': 0.2}


  0%|          | 0/128 [00:00<?, ?it/s]

Train: 288 total batches across 9 algorithms
Val:   36 total batches

Per-algorithm specs:


Done!


TypeError: unsupported format string passed to list.__format__

## Create NAR Model

In [14]:
model = NARModel(
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    processor_type=PROCESSOR_TYPE,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"NAR model parameters: {num_params:,}")

NAR model parameters: 1,587,726


## Training Functions

In [15]:
def train_epoch(model, loader, optimizer, device):
    """Train one epoch over all algorithms (round-robin)."""
    model.train()
    total_loss = 0.0
    algo_losses = {algo: [] for algo in ALGORITHMS}
    n_batches = 0

    for batch, algo, output_types, hint_types in loader:
        batch = batch.to(device)
        spec = loader.specs[algo]
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)

        optimizer.zero_grad()
        output = model(
            inputs=inputs, hints=hints, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        output.total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        loss_val = output.total_loss.item()
        total_loss += loss_val
        algo_losses[algo].append(loss_val)
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    per_algo = {a: np.mean(v) if v else float('nan') for a, v in algo_losses.items()}
    return avg_loss, per_algo


@torch.no_grad()
def validate(model, loader, device):
    """Validate using official CLRS metrics (F1 for masks, accuracy for pointers)."""
    model.eval()
    total_loss = 0.0
    algo_scores = {algo: [] for algo in ALGORITHMS}
    n_batches = 0

    for batch, algo, output_types, hint_types in loader:
        batch = batch.to(device)
        spec = loader.specs[algo]
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)

        output = model(
            inputs=inputs, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        total_loss += output.total_loss.item()
        n_batches += 1

        # Evaluate with CLRS metrics (F1 for masks, accuracy for pointers, etc.)
        evals = evaluate_outputs(output.predictions, outputs, output_types)
        algo_scores[algo].append(evals.get('score', 0.0))

    avg_loss = total_loss / max(n_batches, 1)
    per_algo = {a: np.mean(v) if v else float('nan') for a, v in algo_scores.items()}
    avg_score = np.nanmean(list(per_algo.values()))
    return avg_loss, per_algo, avg_score

## Training Loop

In [16]:
_ckpt_path = CHECKPOINT_DIR / "best.pt"

if _ckpt_path.exists():
    ckpt = torch.load(_ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded existing checkpoint (epoch {ckpt['epoch']}, avg_score={ckpt.get('avg_score', ckpt.get('avg_acc', 'N/A'))})")
    print("Skipping training. Delete checkpoint to retrain.")
    _skip_training = True
else:
    _skip_training = False

if not _skip_training:
    optimizer = optim.AdamW(model.parameters(), lr=NAR_LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=NAR_EPOCHS)

    history = {
        'train_losses': [], 'val_losses': [], 'avg_scores': [],
        'per_algo_loss': [], 'per_algo_score': [],
    }
    best_loss = float('inf')

    for epoch in range(NAR_EPOCHS):
        train_loss, algo_train_loss = train_epoch(model, train_loader, optimizer, device)
        val_loss, algo_val_score, avg_score = validate(model, val_loader, device)
        scheduler.step()

        history['train_losses'].append(train_loss)
        history['val_losses'].append(val_loss)
        history['avg_scores'].append(avg_score)
        history['per_algo_loss'].append(algo_train_loss)
        history['per_algo_score'].append(algo_val_score)

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'epoch': epoch,
                'val_loss': val_loss,
                'avg_score': avg_score,
                'per_algo_score': algo_val_score,
                'config': {
                    'hidden_dim': HIDDEN_DIM, 'num_layers': NUM_LAYERS,
                    'num_heads': NUM_HEADS, 'processor_type': PROCESSOR_TYPE,
                    'algorithms': ALGORITHMS,
                },
            }, CHECKPOINT_DIR / "best.pt")

        if (epoch + 1) % 10 == 0 or epoch == 0:
            algo_str = "  ".join(f"{a[:6]}={algo_val_score.get(a, 0):.2f}" for a in ALGORITHMS[:5])
            print(f"Epoch {epoch+1:3d}/{NAR_EPOCHS} | "
                  f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
                  f"Avg Score: {avg_score:.3f} | {algo_str} ...")

    torch.save({'model_state_dict': model.state_dict()}, CHECKPOINT_DIR / "final.pt")
    torch.save(history, CHECKPOINT_DIR / "training_history.pt")
    print(f"\nBest val loss: {best_loss:.4f}")
    print(f"Saved to {CHECKPOINT_DIR}")

Epoch   1/100 | Train: 2452303.8213 | Val: 4.8265 | Avg Acc: 0.420 | bfs=0.52  dfs=0.12  dijkst=0.39  bellma=0.52  mst_pr=0.37 ...


: 

## Training Curves

In [ ]:
if "history" not in dir():
    history = None

_hist_path = CHECKPOINT_DIR / "training_history.pt"
if _hist_path.exists():
    history = torch.load(_hist_path, weights_only=False)
    print(f"Loaded training history ({len(history['train_losses'])} epochs)")
elif history is None:
    print("No training history available. Run training first.")

if history is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history['train_losses'], label="Train")
    axes[0].plot(history['val_losses'], label="Val")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss (all algorithms)")
    axes[0].legend(); axes[0].set_yscale("log")

    score_key = 'avg_scores' if 'avg_scores' in history else 'avg_accs'
    axes[1].plot(history[score_key])
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("CLRS Score")
    axes[1].set_title("Average Validation Score (CLRS metrics)")
    axes[1].set_ylim(0, 1.05)

    score_algo_key = 'per_algo_score' if 'per_algo_score' in history else 'per_algo_acc'
    if history[score_algo_key]:
        final_scores = history[score_algo_key][-1]
        algos = sorted(final_scores.keys())
        scores = [final_scores[a] for a in algos]
        axes[2].barh(algos, scores, color='steelblue')
        axes[2].set_xlabel("CLRS Score")
        axes[2].set_title("Final Per-Algorithm Score")
        axes[2].set_xlim(0, 1.05)
        for i, v in enumerate(scores):
            axes[2].text(max(v + 0.01, 0.02), i, f"{v:.2f}", va='center', fontsize=9)

    plt.tight_layout()
    plt.show()

## Per-Algorithm Accuracy Over Time

In [ ]:
if history is not None:
    score_algo_key = 'per_algo_score' if 'per_algo_score' in history else 'per_algo_acc'
    if history[score_algo_key]:
        n_algos = len(ALGORITHMS)
        n_cols = min(5, n_algos)
        n_rows = (n_algos + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
        axes = np.array(axes).flatten()

        for i, algo in enumerate(ALGORITHMS):
            scores_over_time = [epoch_scores.get(algo, float('nan'))
                                for epoch_scores in history[score_algo_key]]
            axes[i].plot(scores_over_time)
            axes[i].set_title(algo, fontsize=10)
            axes[i].set_ylim(0, 1.05)
            axes[i].set_xlabel("Epoch")
            axes[i].set_ylabel("CLRS Score")

        for i in range(n_algos, len(axes)):
            axes[i].set_visible(False)

        fig.suptitle("Per-Algorithm Validation Score (CLRS metrics)", fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()

---
## OOD Evaluation (64-node graphs)

The CLRS benchmark evaluates on graphs 4x larger than training.
This tests whether the NAR has learned the algorithm vs memorized patterns.

In [ ]:
# Create OOD validation loaders (64 nodes)
ood_loader = MultiAlgorithmLoader(
    algorithms=ALGORITHMS,
    split="val",
    batch_size=max(1, NAR_BATCH_SIZE // 4),  # smaller batch for larger graphs
    num_samples=VAL_SAMPLES,
    num_nodes=OOD_NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 2,
    shuffle=False,
)
print(f"OOD loader: {len(ood_loader)} batches, {OOD_NUM_NODES}-node graphs")

# Evaluate
ood_loss, ood_per_algo, ood_avg = validate(model, ood_loader, device)
print(f"\nOOD Average Score: {ood_avg:.3f}")
print(f"\nPer-algorithm OOD scores:")
for algo in ALGORITHMS:
    score = ood_per_algo.get(algo, float('nan'))
    print(f"  {algo:25s} {score:.3f}")

# Save OOD results
torch.save({
    'ood_loss': ood_loss,
    'ood_per_algo': ood_per_algo,
    'ood_avg': ood_avg,
    'ood_num_nodes': OOD_NUM_NODES,
}, CHECKPOINT_DIR / "ood_results.pt")
print(f"\nSaved OOD results to {CHECKPOINT_DIR / 'ood_results.pt'}")

## In-Distribution vs OOD Comparison

In [ ]:
# Compare ID (16-node) vs OOD (64-node) scores
if history is not None:
    score_algo_key = 'per_algo_score' if 'per_algo_score' in history else 'per_algo_acc'
    id_scores = history[score_algo_key][-1] if history[score_algo_key] else {}
else:
    id_scores = {}

algos = sorted(set(list(id_scores.keys()) + list(ood_per_algo.keys())))
id_vals = [id_scores.get(a, float('nan')) for a in algos]
ood_vals = [ood_per_algo.get(a, float('nan')) for a in algos]

fig, ax = plt.subplots(figsize=(10, max(4, len(algos) * 0.5)))
y = np.arange(len(algos))
width = 0.35
ax.barh(y - width/2, id_vals, width, label=f'ID ({NUM_NODES} nodes)', color='steelblue')
ax.barh(y + width/2, ood_vals, width, label=f'OOD ({OOD_NUM_NODES} nodes)', color='coral')
ax.set_yticks(y)
ax.set_yticklabels(algos)
ax.set_xlabel("CLRS Score")
ax.set_title("In-Distribution vs OOD Generalization")
ax.legend()
ax.set_xlim(0, 1.05)

for i, (id_v, ood_v) in enumerate(zip(id_vals, ood_vals)):
    if not np.isnan(id_v):
        ax.text(id_v + 0.01, i - width/2, f"{id_v:.2f}", va='center', fontsize=8)
    if not np.isnan(ood_v):
        ax.text(ood_v + 0.01, i + width/2, f"{ood_v:.2f}", va='center', fontsize=8)

plt.tight_layout()
plt.show()

# Print summary table
print(f"\n{'Algorithm':25s} | {'ID Score':>10s} | {'OOD Score':>10s} | {'Drop':>8s}")
print("-" * 60)
for algo, id_v, ood_v in zip(algos, id_vals, ood_vals):
    drop = id_v - ood_v if not (np.isnan(id_v) or np.isnan(ood_v)) else float('nan')
    print(f"{algo:25s} | {id_v:10.3f} | {ood_v:10.3f} | {drop:+8.3f}")

## Summary

The trained NAR checkpoint and evaluation results are saved above.
Use `02_train_sae.ipynb` to collect activations and train SAEs.